# 🎨 ComfyUI trên Google Colab (Free) — model WAI-illustrious

**Cách dùng:**
1. `Runtime` → `Change runtime type` → chọn **T4 GPU** → Save
2. **Dán Civitai API key** vào ô `CIVITAI_KEY` ở Cell 2 (lấy tại civitai.com → Account Settings → API Keys)
3. Chạy lần lượt **Cell 1 → 2 → 3**
4. Copy link ở Cell 3 → **dán vào thanh địa chỉ tab mới** (đừng bấm trực tiếp)
5. Kéo thả file `workflow_noobai_tiengviet.json` vào ComfyUI → nhấn **R** → bấm vào tên model trong node "Tải Model" → chọn **WAI-illustrious** → Queue

**Tính năng:**
- 🥇 Model chính: **WAI-illustrious** (anime đẹp nhất hiện nay, cùng hệ tag Danbooru với NoobAI)
- 💾 Model lưu Google Drive → lần sau KHÔNG tải lại (khởi động ~3 phút)
- 🛡️ Drive lỗi vẫn chạy tiếp được (tự chuyển ổ tạm)
- 🔧 Cell 4 kiểm tra sức khỏe, Cell 5 link dự phòng

⚠️ Cần Drive trống ~7GB. Nếu Drive đã chứa NoobAI (6.9GB) mà sắp đầy: xóa NoobAI (dòng có sẵn trong Cell 2) hoặc dọn bớt Drive. Lần đầu chạy mất ~10 phút (tải model 1 lần duy nhất).

In [ ]:
# ===== CELL 1: Drive + Cài ComfyUI (~2-3 phút) =====
!nvidia-smi --query-gpu=name,memory.total --format=csv

import os

# 1) Kết nối Google Drive (cửa sổ xin quyền -> chọn tài khoản -> CHO PHÉP TẤT CẢ)
#    Nếu Drive lỗi vẫn chạy tiếp bằng ổ tạm (model phải tải lại mỗi phiên)
USE_DRIVE = False
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    os.makedirs('/content/drive/MyDrive/AI_Models/checkpoints', exist_ok=True)
    USE_DRIVE = True
    print('✅ Drive OK — model sẽ được lưu vĩnh viễn.')
except Exception as e:
    print('⚠️ Drive lỗi:', e)
    print('→ Chạy tiếp bằng ổ tạm. Cách sửa Drive: xem ghi chú cuối notebook.')

CKPT_DIR = '/content/drive/MyDrive/AI_Models/checkpoints' if USE_DRIVE else '/content/ComfyUI/models/checkpoints'
with open('/content/ckpt_dir.txt', 'w') as f:
    f.write(CKPT_DIR)

# 2) Cài ComfyUI
os.chdir('/content')
!rm -rf /content/ComfyUI
!git clone https://github.com/comfyanonymous/ComfyUI /content/ComfyUI
os.chdir('/content/ComfyUI')

# Cài thư viện NHƯNG GIỮ NGUYÊN PyTorch-CUDA của Colab
# (để pip tự cài torch sẽ bị đè bản CPU -> lỗi 'Torch not compiled with CUDA enabled')
!grep -viE '^(torch|torchvision|torchaudio)([=<>!~ ]|$)' requirements.txt > /content/req_notorch.txt
!pip install -q -r /content/req_notorch.txt

import torch
assert torch.cuda.is_available(), '❌ PyTorch không thấy GPU! Runtime -> Change runtime type -> T4 GPU, hoặc Restart session rồi chạy lại Cell 1'
print('✅ PyTorch', torch.__version__, '- GPU:', torch.cuda.get_device_name(0))

# 3) Trỏ thư mục model sang Drive (nếu có)
if USE_DRIVE:
    !rm -rf /content/ComfyUI/models/checkpoints
    !ln -s /content/drive/MyDrive/AI_Models/checkpoints /content/ComfyUI/models/checkpoints

print()
print('✅ Xong Cell 1! Thư mục model:', CKPT_DIR)
!ls -lh {CKPT_DIR} 2>/dev/null || echo '(chưa có model - Cell 2 sẽ tải)'

In [ ]:
# ===== CELL 2: Tải Model WAI-illustrious (tự bỏ qua nếu đã có trong Drive) =====
CIVITAI_KEY = ""  # @param {type:"string"}
# ↑ Dán key vào ô bên phải (hoặc giữa 2 dấu nháy). Lấy key: civitai.com -> Account Settings -> API Keys

import os
CKPT_DIR = open('/content/ckpt_dir.txt').read().strip()
print('Model lưu vào:', CKPT_DIR)

WAI = f'{CKPT_DIR}/WAI-illustrious.safetensors'

if os.path.exists(WAI) and os.path.getsize(WAI) > 6_000_000_000:
    print('✅ WAI-illustrious đã có sẵn trong Drive — bỏ qua tải.')
else:
    assert CIVITAI_KEY.strip(), '❌ Chưa dán Civitai API key vào ô CIVITAI_KEY phía trên cell này!'
    !wget -c -O {WAI} "https://civitai.com/api/download/models/2514310?token={CIVITAI_KEY}"
    size = os.path.getsize(WAI) if os.path.exists(WAI) else 0
    if size < 6_000_000_000:
        if os.path.exists(WAI): os.remove(WAI)
        raise RuntimeError('❌ Tải thất bại (file quá nhỏ = Civitai từ chối). Kiểm tra lại key tại civitai.com -> Account Settings -> API Keys, rồi chạy lại cell này.')
    print('✅ Tải xong WAI-illustrious!')

# (TÙY CHỌN) Xóa NoobAI cũ trong Drive để tiết kiệm chỗ - bỏ dấu # nếu muốn
#!rm -f {CKPT_DIR}/noobai-XL-1.1.safetensors

# (TÙY CHỌN) NoobAI-XL 1.1 (6.9 GB) - model anime cũ, không cần key
# Chú ý: Drive free 15GB khó chứa cả 2 model - bỏ dấu # nếu đủ chỗ
#!wget -c -O {CKPT_DIR}/noobai-XL-1.1.safetensors \
#  "https://huggingface.co/Laxhar/noobai-XL-1.1/resolve/main/NoobAI-XL-v1.1.safetensors"

!ls -lh {CKPT_DIR}
print('\n✅ Model sẵn sàng!')

In [ ]:
# ===== CELL 3: Khởi chạy ComfyUI (chạy NỀN) + tạo link truy cập =====
import subprocess, time, socket, re, os

# Dọn tiến trình cũ nếu chạy lại cell
!pkill -f "python main.py" 2>/dev/null || true
!pkill -f cloudflared 2>/dev/null || true
time.sleep(2)

# Cài cloudflared (nếu chưa có)
if not os.path.exists('/usr/local/bin/cloudflared'):
    !wget -q -c https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
    !dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

# 1) Chạy ComfyUI NỀN
os.chdir('/content/ComfyUI')
comfy_log = open('/content/comfyui.log', 'w')
comfy = subprocess.Popen(
    ['python', 'main.py', '--listen', '0.0.0.0', '--enable-cors-header'],
    stdout=comfy_log, stderr=subprocess.STDOUT)
print('⏳ Đang khởi động ComfyUI (30-60 giây)...')

# 2) Đợi cổng 8188 mở
for _ in range(180):
    time.sleep(1)
    if comfy.poll() is not None:
        raise RuntimeError('❌ ComfyUI bị tắt! Xem lỗi: !tail -30 /content/comfyui.log')
    try:
        with socket.create_connection(('127.0.0.1', 8188), timeout=1):
            break
    except OSError:
        pass
else:
    raise RuntimeError('❌ Quá 3 phút chưa mở cổng. Xem log: !tail -30 /content/comfyui.log')
print('✅ ComfyUI đã chạy!')

# 3) Tunnel cloudflared NỀN (http2 - hợp mạng VN)
cf_log = open('/content/cloudflared.log', 'w')
cf = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8188',
     '--http-host-header', '127.0.0.1:8188', '--protocol', 'http2'],
    stdout=cf_log, stderr=subprocess.STDOUT)

# 4) Đọc link
url = None
for _ in range(60):
    time.sleep(1)
    txt = open('/content/cloudflared.log').read()
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', txt)
    if m:
        url = m.group(0)
        break
print()
print('='*60)
if url:
    print('🎨 COPY LINK NÀY, DÁN VÀO THANH ĐỊA CHỈ TAB MỚI:')
    print(url)
else:
    print('⚠️ Chưa lấy được link cloudflare — dùng Cell 5 (localtunnel)')
print('='*60)
print('\n💡 Cell này đã xong nhưng ComfyUI vẫn chạy nền.')
print('   Ảnh nằm ở /content/ComfyUI/output — NHỚ tải ảnh đẹp về máy trước khi ngắt phiên!')


In [ ]:
# ===== CELL 4: KIỂM TRA sức khỏe hệ thống (chạy bất cứ lúc nào) =====
!curl -s -o /dev/null -w "A) ComfyUI noi bo:  HTTP %{http_code} (200 = OK)\n" --max-time 20 http://127.0.0.1:8188/system_stats

import re
txt = open('/content/cloudflared.log').read()
m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', txt)
if m:
    url = m.group(0)
    print('   Link hien tai:', url)
    !curl -s -o /dev/null -w "B) Qua tunnel:      HTTP %{http_code} (200 = OK, 502 = ComfyUI chet, 000 = tunnel chet)\n" --max-time 40 {url}/system_stats
else:
    print('B) Khong tim thay link trong log cloudflared')

print('\n----- LOG ComfyUI (30 dòng cuối) -----')
!tail -30 /content/comfyui.log

In [ ]:
# ===== CELL 5 (DỰ PHÒNG): Link thay thế qua localtunnel =====
# Dùng khi link trycloudflare treo/chặn. ComfyUI phải đang chạy nền (Cell 3 xong).
!npm install -g localtunnel > /dev/null 2>&1

import urllib.request
ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode().strip()
print('='*60)
print('🔑 MẬT KHẨU (Tunnel Password) khi trang hỏi:', ip)
print('='*60)
print('Đợi link https://....loca.lt hiện ra bên dưới rồi mở link đó.\n')

!lt --port 8188

## 📝 Ghi chú

**Khởi động lại phiên mới:** Cell 1 → Cell 2 (tự lướt qua nếu model đã trong Drive) → Cell 3. Tổng ~3 phút.

**Workflow tiếng Việt** (tải từ GitHub repo của bạn, kéo thả vào ComfyUI):
- `workflow_noobai_tiengviet.json` — tạo ảnh anime hàng ngày (khuyên dùng — dùng được cho WAI, chỉ cần đổi model trong dropdown)
- `workflow_animagine_xl_tiengviet.json` — Animagine, cảnh phức tạp
- `workflow_animagine_hiresfix_tiengviet.json` — ảnh nét cao 2 lượt vẽ
- `workflow_inpaint_tiengviet.json` — sửa lỗi / tẩy đồ lạ


**Lưu ảnh:** ảnh nằm ở `/content/ComfyUI/output` (ổ tạm — mất khi ngắt phiên). Tải ảnh đẹp về máy bằng chuột phải → Save image ngay trong ComfyUI, hoặc nén tất cả: `!zip -r /content/anh.zip /content/ComfyUI/output` rồi tải file zip từ thanh Files bên trái.
**Chọn model WAI sau khi kéo workflow vào:** nhấn **R** (refresh) → node "Tải Model (Checkpoint)" → bấm vào tên model → chọn `WAI-illustrious.safetensors`. Chỉ cần làm 1 lần rồi Ctrl+S lưu lại workflow.

**Thông số nhanh:**
- WAI-illustrious / NoobAI: cfg 5, tag chất lượng `masterpiece, best quality, newest, absurdres, highres` (cùng hệ tag Danbooru — prompt cũ dùng lại nguyên xi)
- Animagine: cfg 6, tag `masterpiece, high score, great score, absurdres`
- Tất cả: 832x1216 (dọc) / 1216x832 (ngang), euler_ancestral, steps 26-28
- ⚠️ Key Civitai là của riêng bạn — đừng chia sẻ notebook đã dán key cho người khác

**Xử lý sự cố:**
- Không thấy GPU → Runtime → Change runtime type → T4 GPU
- Link treo/403 → copy link dán vào thanh địa chỉ tab MỚI (đừng bấm trực tiếp)
- 502 → ComfyUI chết: chạy Cell 4 xem log → chạy lại Cell 3
- `Torch not compiled with CUDA` → Runtime → Disconnect and delete runtime → chạy lại từ Cell 1
- Cell 3 báo ComfyUI bị tắt → Runtime → Restart session → Cell 1 → 3 (thường do hết RAM)
- Colab tự ngắt → bình thường với bản free (quota ~3-4h GPU/ngày); model vẫn trong Drive (ảnh trong phiên sẽ mất — nhớ tải về trước), chạy lại 3 phút

## 🔧 Khắc phục lỗi "mount failed" khi kết nối Drive

Làm theo thứ tự, sau mỗi cách chạy lại Cell 1:
1. **Runtime → Disconnect and delete runtime**, rồi chạy lại Cell 1
2. Cửa sổ xin quyền: chọn đúng tài khoản → **"Cho phép tất cả / Select all"** → Continue
3. Cho phép **popup + cookie bên thứ ba** cho colab.research.google.com (🔒 cạnh thanh địa chỉ)
4. Thử cửa sổ ẩn danh, đăng nhập đúng 1 tài khoản Gmail cá nhân
5. Tài khoản trường học/công ty có thể bị chặn Drive → dùng Gmail cá nhân